# UK Road Collisions: exploratory analysis

Inspect data quality, class balance and KSI patterns before training.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from road_severity.data import load_collisions, make_target

DATA_PATH = ROOT / 'data/raw/dft-road-casualty-statistics-collision-last-5-years.csv'
MAX_ROWS = 250_000  # Set to 0 for all rows; sample is stratified by year.
sns.set_theme(style='whitegrid', palette='colorblind')

In [ ]:
frame = load_collisions(DATA_PATH, None if MAX_ROWS == 0 else MAX_ROWS)
frame['ksi'] = make_target(frame)
print(f'Rows: {len(frame):,}; years: {frame.collision_year.min()}-{frame.collision_year.max()}')
frame.head()

## Data quality

Check missingness before selecting features or visual encodings.

In [ ]:
quality = (pd.DataFrame({'missing_count': frame.isna().sum(), 'missing_rate': frame.isna().mean()})
           .sort_values('missing_rate', ascending=False))
quality.head(15)

In [ ]:
severity = frame['collision_severity'].value_counts().sort_index().rename_axis('severity').reset_index(name='collisions')
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=severity, x='severity', y='collisions', ax=ax)
ax.set(title='Collision severity is highly imbalanced', xlabel='Severity code (1 = fatal, 3 = slight)', ylabel='Collisions')
plt.show()

## Temporal patterns

Compare both collision volume and KSI rate.

In [ ]:
by_year = frame.groupby('collision_year', as_index=False).agg(collisions=('collision_index', 'size'), ksi_rate=('ksi', 'mean'))
fig, ax1 = plt.subplots(figsize=(9, 4))
sns.lineplot(data=by_year, x='collision_year', y='collisions', marker='o', ax=ax1, color='#0072B2')
ax1.set(title='Collisions and KSI rate over time', ylabel='Collisions', xlabel='Year')
ax2 = ax1.twinx()
sns.lineplot(data=by_year, x='collision_year', y='ksi_rate', marker='o', ax=ax2, color='#D55E00')
ax2.set_ylabel('KSI rate')
plt.show()
by_year

In [ ]:
hourly = frame.groupby(frame['time'].dt.hour, dropna=True)['ksi'].agg(['mean', 'size']).reset_index(names='hour')
fig, ax = plt.subplots(figsize=(9, 4))
sns.lineplot(data=hourly, x='hour', y='mean', marker='o', ax=ax, color='#D55E00')
ax.set(title='KSI rate by hour of day', xlabel='Hour', ylabel='KSI rate', xticks=range(0, 24, 2))
plt.show()
hourly

## Next step

Record data limitations and candidate risk patterns, then open `02_model_training_evaluation.ipynb`. Associations in these charts are not causal effects.